# Predictive Analytics for Detecting Unauthorized Access in IAM Systems
### Hybrid Random Forest + Isolation Forest + LSTM pipeline (Thesis Chapter 3 methodology → Chapter 4 results)

**⚠️ IMPORTANT — READ BEFORE USING RESULTS IN YOUR THESIS**

This notebook runs end-to-end **out of the box on synthetic demo data** so you can verify the whole
pipeline works before you have the real datasets. The demo data is generated by
`src/data_loader.py` and is **not real IAM data** — its only purpose is to exercise the code.

To produce real, reportable Chapter 4/5 results:
1. Download the real **LANL Cyber1** `auth.txt.gz` (and `redteam.txt.gz`) from
   https://csr.lanl.gov/data/cyber1/ and place them in `data/`.
2. Obtain or generate real **IAM-SynthSim** (or an equivalent agent-based IAM simulator) logs and place
   the CSV in `data/`.
3. In the **Setup** cell below, set `USE_REAL_DATA = True` and point the paths at your files.
4. Re-run the whole notebook. Every table and plot below will then reflect your real experiment,
   and can be copied directly into Chapter 4.

See `README.md` for full data-acquisition instructions.

In [ ]:
import sys, warnings
sys.path.insert(0, "src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_loader as dl
import feature_engineering as fe
import preprocessing as pp
import models as mdl
import evaluate as ev

np.random.seed(42)
plt.rcParams["figure.figsize"] = (6, 4)

## Setup

Set `USE_REAL_DATA = True` and fill in the paths once you have downloaded the real datasets
(see README.md). Leave as `False` to run on synthetic demo data.

In [ ]:
USE_REAL_DATA = False

LANL_AUTH_PATH = "data/auth.txt.gz"       # only used if USE_REAL_DATA=True
LANL_REDTEAM_PATH = "data/redteam.txt.gz" # only used if USE_REAL_DATA=True
SYNTHSIM_PATH = "data/iam_synthsim.csv"           # only used if USE_REAL_DATA=True

N_DEMO_EVENTS = 15_000   # size of synthetic demo data (only used if USE_REAL_DATA=False)
SEQ_LEN = 24             # 24-hour sequence window (Section 3.6)

## Section 4.1 — Data Loading and Descriptive Statistics

Loads both data sources, engineers the common feature schema (Section 3.5), and reports class
balance before/after preprocessing.

In [ ]:
if USE_REAL_DATA:
    lanl_raw = dl.load_lanl_cyber1(path=LANL_AUTH_PATH, redteam_path=LANL_REDTEAM_PATH)
    synthsim_raw = dl.load_iam_synthsim(path=SYNTHSIM_PATH)
else:
    lanl_raw = dl.load_lanl_cyber1(n_rows=N_DEMO_EVENTS)
    synthsim_raw = dl.load_iam_synthsim(n_events=N_DEMO_EVENTS)

lanl_feat = fe.build_common_features_lanl(lanl_raw)
synthsim_feat = fe.build_common_features_synthsim(synthsim_raw)

print("LANL common-feature table:", lanl_feat.shape)
print("IAM-SynthSim common-feature table:", synthsim_feat.shape)

In [ ]:
def class_balance_table(feat_df, name):
    n = len(feat_df)
    n_pos = int(feat_df["label"].sum())
    return pd.DataFrame({
        "dataset": [name],
        "n_events": [n],
        "n_threat_events": [n_pos],
        "threat_ratio": [round(n_pos / n, 5)],
    })

balance = pd.concat([
    class_balance_table(lanl_feat, "LANL Cyber1"),
    class_balance_table(synthsim_feat, "IAM-SynthSim"),
]).set_index("dataset")
balance

## Section 4.2 — Hybrid Model vs. Baselines (RQ1)

Trains a rule-based baseline, each individual model component, and the full hybrid ensemble on
the LANL dataset, then compares them on the held-out test set (which keeps the real class
imbalance — see `preprocessing.stratified_split`).

In [ ]:
def rule_based_baseline(X_test_raw, feature_names):
    """Simple fixed-threshold baseline representative of Section 2.1
    (e.g. flag if login_velocity or failure_rate exceeds a static cutoff)."""
    df = pd.DataFrame(X_test_raw, columns=feature_names)
    flagged = (
        (df["login_velocity"] > df["login_velocity"].quantile(0.99)) |
        (df["failure_rate"] > 0.5) |
        (df["privilege_change"] == 1)
    )
    return flagged.astype(int).values

In [ ]:
X_static = lanl_feat[fe.STATIC_FEATURES].values
y = lanl_feat["label"].values

X_train, X_val, X_test, y_train, y_val, y_test = pp.stratified_split(X_static, y)
X_train_s, X_val_s, X_test_s, scaler = pp.scale_features(X_train, X_val, X_test)
X_train_res, y_train_res = pp.resample_training_set(X_train_s, y_train)

X_seq, y_seq = fe.create_sequences(lanl_feat, seq_len=SEQ_LEN)
from sklearn.model_selection import train_test_split as tts
Xseq_train, Xseq_test, yseq_train, yseq_test = tts(
    X_seq, y_seq, test_size=0.3, stratify=y_seq, random_state=42
)

print("Train:", X_train_res.shape, "Val:", X_val_s.shape, "Test:", X_test_s.shape)
print("Sequences -> train:", Xseq_train.shape, "test:", Xseq_test.shape)

In [ ]:
results = []

# --- Rule-based baseline ---
y_pred_rule = rule_based_baseline(X_test, fe.STATIC_FEATURES)
results.append(ev.compute_metrics(y_test, y_pred_rule, y_pred_rule.astype(float), "Rule-based baseline"))

# --- Random Forest ---
rf = mdl.build_random_forest()
rf.fit(X_train_res, y_train_res)
p_rf = rf.predict_proba(X_test_s)[:, 1]
results.append(ev.compute_metrics(y_test, (p_rf >= 0.5).astype(int), p_rf, "Random Forest"))

# --- Isolation Forest ---
iforest = mdl.build_isolation_forest()
iforest.fit(X_train_s[y_train == 0])
p_if = mdl.isolation_forest_scores(iforest, X_test_s)
results.append(ev.compute_metrics(y_test, (p_if >= 0.5).astype(int), p_if, "Isolation Forest"))

# --- LSTM ---
lstm = mdl.build_lstm(Xseq_train.shape[1], Xseq_train.shape[2])
lstm.fit(Xseq_train, yseq_train, epochs=3, batch_size=512, validation_split=0.1, verbose=0)
p_lstm = lstm.predict(Xseq_test, verbose=0).ravel()
results.append(ev.compute_metrics(yseq_test, (p_lstm >= 0.5).astype(int), p_lstm, "LSTM"))

results_df = ev.results_table(results)
results_df

### Hybrid ensemble

The hybrid ensemble needs aligned static + sequence inputs for the *same* held-out rows, so it is
trained/evaluated on a matching split below (Section 3.6: soft-voting weights RF=0.4, iForest=0.3,
LSTM=0.3).

In [ ]:
lanl_feat_reset = lanl_feat.reset_index(drop=True)
X_static_all = lanl_feat_reset[fe.STATIC_FEATURES].values
y_all = lanl_feat_reset["label"].values
X_seq_all, y_seq_all = fe.create_sequences(lanl_feat_reset, seq_len=SEQ_LEN, stride=1)

# align: create_sequences preserves per-user row order grouped, not the original row order,
# so for a fair hybrid comparison we rebuild static features in the SAME per-user order.
aligned_static = []
for _, g in lanl_feat_reset.groupby("user_id"):
    aligned_static.append(g[fe.STATIC_FEATURES].to_numpy(dtype=float))
X_static_aligned = np.vstack(aligned_static)

idx_train, idx_test = tts(np.arange(len(y_seq_all)), test_size=0.3,
                           stratify=y_seq_all, random_state=42)

Xs_tr, Xs_te = X_static_aligned[idx_train], X_static_aligned[idx_test]
Xq_tr, Xq_te = X_seq_all[idx_train], X_seq_all[idx_test]
y_tr, y_te = y_seq_all[idx_train], y_seq_all[idx_test]

Xs_tr_scaled = scaler.transform(Xs_tr)
Xs_te_scaled = scaler.transform(Xs_te)
Xs_tr_res, y_tr_res = pp.resample_training_set(Xs_tr_scaled, y_tr)

hybrid = mdl.HybridEnsemble(weights=(0.4, 0.3, 0.3))
hybrid.rf.fit(Xs_tr_res, y_tr_res)
hybrid.iforest.fit(Xs_tr_scaled[y_tr == 0])
hybrid.lstm = mdl.build_lstm(Xq_tr.shape[1], Xq_tr.shape[2])
hybrid.lstm.fit(Xq_tr, y_tr, epochs=3, batch_size=512, validation_split=0.1, verbose=0)

p_hybrid = hybrid.predict_proba(Xs_te_scaled, Xq_te)
hybrid_metrics = ev.compute_metrics(y_te, (p_hybrid >= 0.5).astype(int), p_hybrid, "Hybrid Ensemble")
results_df_full = ev.results_table(results + [hybrid_metrics])
results_df_full

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ev.plot_confusion_matrix(y_te, (p_hybrid >= 0.5).astype(int), "Hybrid Ensemble — Confusion Matrix", ax=axes[0])
ev.plot_roc(y_te, p_hybrid, label="Hybrid Ensemble", ax=axes[1])
ev.plot_roc(y_test, p_rf, label="Random Forest", ax=axes[1])
ev.plot_roc(y_test, p_if, label="Isolation Forest", ax=axes[1])
plt.tight_layout()
plt.show()

## Section 4.3 — Temporal Ablation (RQ2)

Compares the LSTM's sequential (24-hour window) representation against an equivalent **flattened**
representation of the same events (last event's static features only, no sequence context).

In [ ]:
# Sequential (already trained above): p_lstm / yseq_test
seq_metrics = ev.compute_metrics(yseq_test, (p_lstm >= 0.5).astype(int), p_lstm, "LSTM (sequential)")

# Flattened equivalent: use only the LAST row of each sequence window as a static feature vector
X_flat = Xseq_test[:, -1, :]
rf_flat = mdl.build_random_forest()
X_flat_train = Xseq_train[:, -1, :]
rf_flat.fit(X_flat_train, yseq_train)
p_flat = rf_flat.predict_proba(X_flat)[:, 1]
flat_metrics = ev.compute_metrics(yseq_test, (p_flat >= 0.5).astype(int), p_flat, "Flattened (static-only)")

ablation_table = ev.results_table([flat_metrics, seq_metrics])
ablation_table

## Section 4.4 — Cross-Dataset Generalization (RQ3)

Trains the Random Forest component on LANL and tests on IAM-SynthSim, and vice versa, comparing against
same-dataset performance to quantify the generalization gap.

In [ ]:
def rf_train_test(train_feat, test_feat):
    Xtr = train_feat[fe.STATIC_FEATURES].values
    ytr = train_feat["label"].values
    Xte = test_feat[fe.STATIC_FEATURES].values
    yte = test_feat["label"].values

    Xtr_tr, Xtr_val, Xtr_te, ytr_tr, ytr_val, ytr_te = pp.stratified_split(Xtr, ytr)
    Xtr_s, _, _, sc = pp.scale_features(Xtr_tr, Xtr_val, Xtr_te)
    Xtr_res, ytr_res = pp.resample_training_set(Xtr_s, ytr_tr)

    model = mdl.build_random_forest()
    model.fit(Xtr_res, ytr_res)

    Xte_s = sc.transform(Xte)
    p = model.predict_proba(Xte_s)[:, 1]
    return ev.compute_metrics(yte, (p >= 0.5).astype(int), p, "cross")

same_lanl = rf_train_test(lanl_feat, lanl_feat)
cross_lanl_to_synthsim = rf_train_test(lanl_feat, synthsim_feat)
same_synthsim = rf_train_test(synthsim_feat, synthsim_feat)
cross_synthsim_to_lanl = rf_train_test(synthsim_feat, lanl_feat)

rq3_table = pd.DataFrame([
    {"setting": "Train LANL / Test LANL (same-dataset)", **same_lanl},
    {"setting": "Train LANL / Test IAM-SynthSim (cross-dataset)", **cross_lanl_to_synthsim},
    {"setting": "Train IAM-SynthSim / Test IAM-SynthSim (same-dataset)", **same_synthsim},
    {"setting": "Train IAM-SynthSim / Test LANL (cross-dataset)", **cross_synthsim_to_lanl},
]).set_index("setting").round(4)
rq3_table

## Section 4.5 — SHAP Feature Importance

Identifies which behavioral, contextual, and temporal features contribute most to the Random
Forest component's decisions.

In [ ]:
importance = ev.shap_feature_importance(rf, X_test_s, fe.STATIC_FEATURES, max_display=10)
importance

In [ ]:
plt.figure(figsize=(6, 4))
plt.barh(importance["feature"][::-1], importance["mean_abs_shap"][::-1])
plt.xlabel("Mean |SHAP value|")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

## Exporting results for Chapter 4

Run this cell once you've switched to real data to export all tables as CSV files you can paste
into Word / format as thesis tables.

In [ ]:
import os
os.makedirs("results", exist_ok=True)
balance.to_csv("results/4_1_class_balance.csv")
results_df_full.to_csv("results/4_2_model_comparison.csv")
ablation_table.to_csv("results/4_3_temporal_ablation.csv")
rq3_table.to_csv("results/4_4_cross_dataset.csv")
importance.to_csv("results/4_5_feature_importance.csv", index=False)
print("Saved CSV tables to results/")